# Radio France Podcast Transcriber (GPU)

Ce notebook lance un serveur de transcription sur le GPU gratuit de Google Colab.

## Instructions
1. **Runtime > Change runtime type > T4 GPU**
2. **Run All** (Ctrl+F9)
3. Copiez l'URL ngrok affichee
4. Collez-la dans l'application Radio France Summarizer

Le serveur restera actif tant que le notebook tourne (~90 min en gratuit).

In [ ]:
# 1. Installation des dependances
!pip install -q faster-whisper flask pyngrok

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# 2. Chargement du modele Whisper sur GPU
from faster_whisper import WhisperModel
import time

MODEL_SIZE = "large-v3"  # Meilleure qualite possible — rapide sur GPU

print(f"Chargement du modele '{MODEL_SIZE}' sur GPU...")
start = time.time()
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")
print(f"Modele charge en {time.time() - start:.1f}s")

In [ ]:
# 3. Serveur de transcription
import os
import re
import json
import tempfile
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL_SIZE, "device": "cuda"})

@app.route("/transcribe", methods=["POST"])
def transcribe():
    if "file" not in request.files:
        return jsonify({"error": "No file uploaded"}), 400

    audio_file = request.files["file"]
    language = request.form.get("language", "fr")

    # Sauvegarder temporairement
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
        audio_file.save(tmp.name)
        tmp_path = tmp.name

    try:
        start = time.time()
        segments, info = model.transcribe(
            tmp_path,
            language=language,
            beam_size=5,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=500),
        )

        parts = [seg.text.strip() for seg in segments]
        transcript = " ".join(parts)
        transcript = re.sub(r" +", " ", transcript)
        elapsed = time.time() - start

        return jsonify({
            "transcript": transcript,
            "duration": round(info.duration, 1),
            "processing_time": round(elapsed, 1),
            "speedup": round(info.duration / elapsed, 1) if elapsed > 0 else 0,
            "language": info.language,
            "model": MODEL_SIZE,
        })
    finally:
        os.unlink(tmp_path)

print("Serveur Flask pret.")

In [ ]:
# 4. Exposer le serveur via ngrok
from pyngrok import ngrok
import threading

# Lancer Flask dans un thread
threading.Thread(target=lambda: app.run(port=5000), daemon=True).start()

# Creer le tunnel ngrok
public_url = ngrok.connect(5000).public_url

print("\n" + "=" * 60)
print(f"  SERVEUR DE TRANSCRIPTION ACTIF")
print(f"  URL : {public_url}")
print(f"  Modele : {MODEL_SIZE} sur GPU")
print("=" * 60)
print(f"\nCollez cette URL dans l'application Radio France Summarizer.")
print(f"Le serveur reste actif tant que ce notebook tourne.")